In [2]:
!pip install flask nest_asyncio



mambajs 0.19.13

Process pip requirements ...

Requirement jinja2 already satisfied.
Requirement markupsafe already satisfied.


In [4]:
import datetime
from IPython.display import display, clear_output
import ipywidgets as widgets
import time
import threading

# 1. In-memory storage for the tracker
tasks = [
    {"id": 1, "title": "Organic Chem Revision", "subject": "Chemistry", "status": "Pending"},
    {"id": 2, "title": "Electromagnetism Numerical Set", "subject": "Physics", "status": "Completed"}
]
study_logs = []

# 2. UI Components & Layouts
output_area = widgets.Output()

# Stats display
stats_html = widgets.HTML()

# Add Task form elements
task_title_input = widgets.Text(placeholder='Task description...', description='Task:')
task_subject_input = widgets.Text(placeholder='e.g., Physics', description='Subject:')
add_task_btn = widgets.Button(description='Add Task', button_style='primary', icon='plus')

# Log Study form elements
log_subject_input = widgets.Text(placeholder='Subject', description='Subject:')
log_duration_input = widgets.IntText(value=25, description='Mins:')
log_session_btn = widgets.Button(description='Log Session', button_style='info', icon='history')

# Pomodoro Timer components
timer_label = widgets.Label(value="25:00", layout=widgets.Layout(justify_content='center'))
timer_label.style.font_size = '32px'
timer_label.style.font_weight = 'bold'
start_timer_btn = widgets.Button(description='Start', button_style='success', icon='play')
pause_timer_btn = widgets.Button(description='Pause', button_style='warning', icon='pause')

# Global control flags for timer thread
timer_seconds = 1500
timer_running = False

# 3. Logic & Functions
def calculate_stats():
    completed = sum(1 for t in tasks if t['status'] == 'Completed')
    pending = sum(1 for t in tasks if t['status'] == 'Pending')
    total_hours = sum(log['duration'] for log in study_logs) / 60 if study_logs else 0
    
    stats_html.value = f"""
    <div style="background-color: #1e1e1e; padding: 15px; border-radius: 8px; color: white; margin-bottom: 15px; font-family: sans-serif;">
        <h2 style="color: #6200ee; text-align: center; margin-top:0;">Academic Tracker Dashboard</h2>
        <table style="width: 100%; text-align: center; border-collapse: collapse;">
            <tr>
                <td style="padding: 10px; border-right: 1px solid #444;">
                    <div style="font-size: 14px; color: #aaa;">Pending Tasks</div>
                    <div style="font-size: 24px; font-weight: bold; color: #ffc107;">{pending}</div>
                </td>
                <td style="padding: 10px; border-right: 1px solid #444;">
                    <div style="font-size: 14px; color: #aaa;">Completed Tasks</div>
                    <div style="font-size: 24px; font-weight: bold; color: #28a745;">{completed}</div>
                </td>
                <td style="padding: 10px;">
                    <div style="font-size: 14px; color: #aaa;">Total Study Hours</div>
                    <div style="font-size: 24px; font-weight: bold; color: #17a2b8;">{total_hours:.2f} hrs</div>
                </td>
            </tr>
        </table>
    </div>
    """

def render_task_list():
    with output_area:
        clear_output(wait=True)
        calculate_stats()
        display(stats_html)
        
        print("📋 CURRENT TASKS:")
        print("-" * 50)
        if not tasks:
            print(" No tasks left! Add one below.")
        for t in tasks:
            status_mark = "✅ [Completed]" if t['status'] == 'Completed' else "⏳ [Pending]"
            print(f"ID {t['id']}: {status_mark} ({t['subject']}) {t['title']}")
        print("-" * 50)

def on_add_task_clicked(b):
    if task_title_input.value and task_subject_input.value:
        new_id = max([t['id'] for t in tasks], default=0) + 1
        tasks.append({
            "id": new_id, 
            "title": task_title_input.value, 
            "subject": task_subject_input.value, 
            "status": "Pending"
        })
        task_title_input.value = ''
        task_subject_input.value = ''
        render_task_list()

def on_log_session_clicked(b):
    if log_subject_input.value and log_duration_input.value > 0:
        study_logs.append({
            "date": datetime.date.today().strftime("%Y-%m-%d"),
            "subject": log_subject_input.value,
            "duration": int(log_duration_input.value)
        })
        log_subject_input.value = ''
        render_task_list()

# Action buttons to resolve tasks directly via inputs
resolve_input = widgets.IntText(description="Task ID:", layout=widgets.Layout(width='150px'))
toggle_btn = widgets.Button(description="Check/Uncheck", button_style='warning')
delete_btn = widgets.Button(description="Delete Task", button_style='danger')

def on_toggle_clicked(b):
    for t in tasks:
        if t['id'] == resolve_input.value:
            t['status'] = 'Completed' if t['status'] == 'Pending' else 'Pending'
            break
    render_task_list()

def on_delete_clicked(b):
    global tasks
    tasks = [t for t in tasks if t['id'] != resolve_input.value]
    render_task_list()

# Async background worker thread for safe notebook timing operations
def timer_worker():
    global timer_seconds, timer_running
    while timer_running and timer_seconds > 0:
        time.sleep(1)
        if timer_running: # Secondary check if paused mid-sleep
            timer_seconds -= 1
            mins, secs = divmod(timer_seconds, 60)
            timer_label.value = f"{mins:02d}:{secs:02d}"
    if timer_seconds == 0:
        timer_label.value = "00:00 - Break Time!"

def on_start_timer(b):
    global timer_running
    if not timer_running:
        timer_running = True
        t = threading.Thread(target=timer_worker)
        t.start()

def on_pause_timer(b):
    global timer_running
    timer_running = False

# Wire up event hooks
add_task_btn.on_click(on_add_task_clicked)
log_session_btn.on_click(on_log_session_clicked)
toggle_btn.on_click(on_toggle_clicked)
delete_btn.on_click(on_delete_clicked)
start_timer_btn.on_click(on_start_timer)
pause_timer_btn.on_click(on_pause_timer)

# 4. Assemble Layout Views
task_form = widgets.VBox([
    widgets.HTML("<h4>Add New Task</h4>"),
    task_title_input, task_subject_input, add_task_btn
])

task_actions = widgets.VBox([
    widgets.HTML("<h4>Manage Status by ID</h4>"),
    widgets.HBox([resolve_input, toggle_btn, delete_btn])
])

study_form = widgets.VBox([
    widgets.HTML("<h4>Log Study Time Manually</h4>"),
    log_subject_input, log_duration_input, log_session_btn
])

timer_box = widgets.VBox([
    widgets.HTML("<h4 style='text-align:center;'>Pomodoro Timer</h4>"),
    timer_label,
    widgets.HBox([start_timer_btn, pause_timer_btn], layout=widgets.Layout(justify_content='center'))
], layout=widgets.Layout(border='1px solid #444', padding='10px', border_radius='8px'))

# Render application UI
render_task_list()
display(output_area)
display(widgets.HBox([widgets.VBox([task_form, task_actions]), widgets.VBox([timer_box, study_form])], layout=widgets.Layout(gap='40px')))

Output()

In [8]:
import datetime
from IPython.display import display, clear_output
import ipywidgets as widgets
import time
import threading
import matplotlib.pyplot as plt

# Core data storage
tasks = [
    {"id": 1, "title": "Organic Chem Revision", "subject": "Chemistry", "status": "Pending"},
    {"id": 2, "title": "Electromagnetism Numerical Set", "subject": "Physics", "status": "Completed"}
]
study_logs = [
    {"date": "2026-05-15", "subject": "Physics", "duration": 60},
    {"date": "2026-05-16", "subject": "Chemistry", "duration": 45},
    {"date": "2026-05-17", "subject": "Data Science", "duration": 120}
]

print("✅ Step 1 complete: Storage initialized successfully!")

✅ Step 1 complete: Storage initialized successfully!


In [9]:
output_area = widgets.Output()
chart_output = widgets.Output()
stats_html = widgets.HTML()

def update_chart():
    with chart_output:
        clear_output(wait=True)
        if not study_logs:
            return
        subject_totals = {}
        for log in study_logs:
            sub = log['subject'].capitalize()
            subject_totals[sub] = subject_totals.get(sub, 0) + (log['duration'] / 60)
            
        plt.style.use('dark_background')
        fig, ax = plt.subplots(figsize=(4, 2.5))
        subjects = list(subject_totals.keys())
        hours = list(subject_totals.values())
        ax.bar(subjects, hours, color=['#6200ee', '#03dac6', '#ffb74d', '#ff5722'][:len(subjects)])
        ax.set_ylabel('Hours')
        ax.set_title('Study Distribution', fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.show()

def calculate_stats():
    completed = sum(1 for t in tasks if t['status'] == 'Completed')
    pending = sum(1 for t in tasks if t['status'] == 'Pending')
    total_hours = sum(log['duration'] for log in study_logs) / 60 if study_logs else 0
    stats_html.value = f"""
    <div style="background-color: #1e1e1e; padding: 12px; border-radius: 8px; color: white; margin-bottom: 10px; font-family: sans-serif; border: 1px solid #333;">
        <h3 style="color: #6200ee; text-align: center; margin-top:0;">Academic Tracker Dashboard</h3>
        <table style="width: 100%; text-align: center;">
            <tr>
                <td><div style="color: #aaa; font-size:12px;">Pending</div><div style="font-size: 20px; font-weight: bold; color: #ffc107;">{pending}</div></td>
                <td><div style="color: #aaa; font-size:12px;">Completed</div><div style="font-size: 20px; font-weight: bold; color: #28a745;">{completed}</div></td>
                <td><div style="color: #aaa; font-size:12px;">Total Hours</div><div style="font-size: 20px; font-weight: bold; color: #17a2b8;">{total_hours:.2f}</div></td>
            </tr>
        </table>
    </div>
    """

def render_task_list():
    with output_area:
        clear_output(wait=True)
        calculate_stats()
        display(stats_html)
        print("📋 CURRENT TASKS:\n" + "-"*40)
        for t in tasks:
            mark = "✅ [Done]" if t['status'] == 'Completed' else "⏳ [Wait]"
            print(f"ID {t['id']}: {mark} ({t['subject']}) {t['title']}")
        print("-"*40)
    update_chart()

print("✅ Step 2 complete: Dashboard engine configured!")

✅ Step 2 complete: Dashboard engine configured!


In [10]:
# Create Controls
task_title_input = widgets.Text(placeholder='Task description...', description='Task:')
task_subject_input = widgets.Text(placeholder='e.g., Physics', description='Subject:')
add_task_btn = widgets.Button(description='Add Task', button_style='primary', icon='plus')

log_subject_input = widgets.Text(placeholder='Subject', description='Subject:')
log_duration_input = widgets.IntText(value=25, description='Mins:')
log_session_btn = widgets.Button(description='Log Session', button_style='info', icon='history')

resolve_input = widgets.IntText(description="Task ID:", layout=widgets.Layout(width='120px'))
toggle_btn = widgets.Button(description="Toggle Status", button_style='warning')
delete_btn = widgets.Button(description="Delete", button_style='danger')

# Form Button Events
def on_add_task_clicked(b):
    if task_title_input.value and task_subject_input.value:
        new_id = max([t['id'] for t in tasks], default=0) + 1
        tasks.append({"id": new_id, "title": task_title_input.value, "subject": task_subject_input.value, "status": "Pending"})
        task_title_input.value, task_subject_input.value = '', ''
        render_task_list()

def on_log_session_clicked(b):
    if log_subject_input.value and log_duration_input.value > 0:
        study_logs.append({"date": datetime.date.today().strftime("%Y-%m-%d"), "subject": log_subject_input.value, "duration": int(log_duration_input.value)})
        log_subject_input.value = ''
        render_task_list()

def on_toggle_clicked(b):
    for t in tasks:
        if t['id'] == resolve_input.value:
            t['status'] = 'Completed' if t['status'] == 'Pending' else 'Pending'
    render_task_list()

def on_delete_clicked(b):
    global tasks
    tasks = [t for t in tasks if t['id'] != resolve_input.value]
    render_task_list()

add_task_btn.on_click(on_add_task_clicked)
log_session_btn.on_click(on_log_session_clicked)
toggle_btn.on_click(on_toggle_clicked)
delete_btn.on_click(on_delete_clicked)

# Render View Layout
form_box = widgets.VBox([widgets.HTML("<b>Add Task</b>"), task_title_input, task_subject_input, add_task_btn, widgets.HTML("<br><b>Manage Task ID</b>"), widgets.HBox([resolve_input, toggle_btn, delete_btn])])
timer_and_log = widgets.VBox([widgets.HTML("<b>Log Work</b>"), log_subject_input, log_duration_input, log_session_btn])

render_task_list()
display(output_area)
display(widgets.HBox([form_box, timer_and_log, chart_output]))

Output()

In [11]:
# Create Controls
task_title_input = widgets.Text(placeholder='Task description...', description='Task:')
task_subject_input = widgets.Text(placeholder='e.g., Physics', description='Subject:')
add_task_btn = widgets.Button(description='Add Task', button_style='primary', icon='plus')

log_subject_input = widgets.Text(placeholder='Subject', description='Subject:')
log_duration_input = widgets.IntText(value=25, description='Mins:')
log_session_btn = widgets.Button(description='Log Session', button_style='info', icon='history')

resolve_input = widgets.IntText(description="Task ID:", layout=widgets.Layout(width='120px'))
toggle_btn = widgets.Button(description="Toggle Status", button_style='warning')
delete_btn = widgets.Button(description="Delete", button_style='danger')

# Form Button Events
def on_add_task_clicked(b):
    if task_title_input.value and task_subject_input.value:
        new_id = max([t['id'] for t in tasks], default=0) + 1
        tasks.append({"id": new_id, "title": task_title_input.value, "subject": task_subject_input.value, "status": "Pending"})
        task_title_input.value, task_subject_input.value = '', ''
        render_task_list()

def on_log_session_clicked(b):
    if log_subject_input.value and log_duration_input.value > 0:
        study_logs.append({"date": datetime.date.today().strftime("%Y-%m-%d"), "subject": log_subject_input.value, "duration": int(log_duration_input.value)})
        log_subject_input.value = ''
        render_task_list()

def on_toggle_clicked(b):
    for t in tasks:
        if t['id'] == resolve_input.value:
            t['status'] = 'Completed' if t['status'] == 'Pending' else 'Pending'
    render_task_list()

def on_delete_clicked(b):
    global tasks
    tasks = [t for t in tasks if t['id'] != resolve_input.value]
    render_task_list()

add_task_btn.on_click(on_add_task_clicked)
log_session_btn.on_click(on_log_session_clicked)
toggle_btn.on_click(on_toggle_clicked)
delete_btn.on_click(on_delete_clicked)

# Render View Layout
form_box = widgets.VBox([widgets.HTML("<b>Add Task</b>"), task_title_input, task_subject_input, add_task_btn, widgets.HTML("<br><b>Manage Task ID</b>"), widgets.HBox([resolve_input, toggle_btn, delete_btn])])
timer_and_log = widgets.VBox([widgets.HTML("<b>Log Work</b>"), log_subject_input, log_duration_input, log_session_btn])

render_task_list()
display(output_area)
display(widgets.HBox([form_box, timer_and_log, chart_output]))

Output(outputs=({'data': {'application/vnd.jupyter.widget-view+json': {'model_id': '56272409423d48009dec191335…

In [12]:
# 1. Setup Configuration & Goal State
DAILY_GOAL_HOURS = 3.0  # Set your study goal here (e.g., 3 hours)

goal_output = widgets.Output()
progress_bar = widgets.FloatProgress(
    value=0.0,
    min=0.0,
    max=DAILY_GOAL_HOURS,
    description='Day Goal:',
    bar_style='info', # 'success', 'info', 'warning', 'danger' or ''
    orientation='horizontal',
    layout=widgets.Layout(width='90%', height='25px')
)

goal_text = widgets.HTML()

# 2. Logic to update the progress bar view
def update_goal_progress():
    with goal_output:
        clear_output(wait=True)
        
        # Calculate total hours logged
        total_hours = sum(log['duration'] for log in study_logs) / 60 if study_logs else 0
        
        # Update progress bar value (clamp to maximum goal)
        progress_bar.value = min(total_hours, DAILY_GOAL_HOURS)
        
        # Dynamically shift colors based on milestone completion
        if total_hours >= DAILY_GOAL_HOURS:
            progress_bar.bar_style = 'success'
            message = f"<b style='color:#28a745;'>🎉 Daily Goal Achieved! ({total_hours:.2f} / {DAILY_GOAL_HOURS} hrs)</b>"
        elif total_hours >= (DAILY_GOAL_HOURS / 2):
            progress_bar.bar_style = 'warning'
            remaining = DAILY_GOAL_HOURS - total_hours
            message = f"<span style='color:#ffc107;'>Keep going! Over halfway there. ({remaining:.2f} hrs left)</span>"
        else:
            progress_bar.bar_style = 'info'
            remaining = DAILY_GOAL_HOURS - total_hours
            message = f"<span style='color:#17a2b8;'>Starting strong! Need {remaining:.2f} more hours today.</span>"
            
        goal_text.value = f"<div style='font-family: sans-serif; font-size: 14px; margin-top: 5px;'>{message}</div>"
        
        # Render the integrated widgets inside this container block
        display(widgets.VBox([progress_bar, goal_text]))

# 3. Inject our new feature into the existing logging engine dynamically
# This intercepts when you click 'Log Session' in Cell 3 and automatically refreshes the bar
base_log_handler = log_session_btn._click_handlers.callbacks[0] if log_session_btn._click_handlers.callbacks else None

def wrapper_log_handler(b):
    if base_log_handler:
        base_log_handler(b) # Run original log logic
    update_goal_progress()  # Run our new progress bar calculation

# Swap the event listener to the upgraded wrapper version
log_session_btn._click_handlers.callbacks.clear()
log_session_btn.on_click(wrapper_log_handler)

# Initial render display
display(widgets.HTML("<hr><h4>🎯 Daily Target Tracking</h4>"))
display(goal_output)
update_goal_progress()

HTML(value='<hr><h4>🎯 Daily Target Tracking</h4>')

Output()